<a href="https://colab.research.google.com/github/lampe142/BASEforHANK_Cloud_CPU_GPU_TPU/blob/master/reactant_example_compute_pi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Colab Test

get the git lab connected to the colab session

In [ ]:
6-4

2

In [1]:
using Pkg

In [2]:
println("Julia Version Running: ", VERSION)

Julia Version Running: 1.12.6


In [3]:
readdir()

2-element Vector{String}:
 ".config"
 "sample_data"

In [4]:
Pkg.add("DataFrames")
Pkg.add("BenchmarkTools")
Pkg.add("Reactant")
Pkg.add("Random")
Pkg.add("JLD2")
Pkg.add("CSV")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
   Installed BenchmarkTools ─ v1.8.0
    Updating `~/.julia/environments/v1.12/Project.toml`
  [6e4b80f9] + BenchmarkTools v1.8.0
    Updating `~/.julia/environments/v1.12/Manifest.toml`
  [6e4b80f9] + BenchmarkTools v1.8.0
  [9abbd945] + Profile v1.11.0
Precompiling packages...
   6198.6 ms  ✓ BenchmarkTools
  1 dependency successfully precompiled in 7 seconds. 481 already precompiled.
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
    Updating `~/.julia/environments/v1.12/Pr

In [5]:
import Pkg, Reactant, BenchmarkTools, Random, JLD2, CSV, DataFrames, Dates

In [6]:
using BenchmarkTools

In [7]:
using Reactant
Reactant.set_default_backend("tpu")

┌ Warning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
└ @ Reactant.Accelerators.TPU ~/.julia/packages/Reactant/H58gu/src/accelerators/TPU.jl:240
I0000 00:00:1777019941.195834     339 pjrt_api.cc:118] GetPjrtApi was found for tpu at /root/.julia/scratchspaces/3c362404-f566-11ee-1572-e11a4b42c853/libtpu/libtpu-0_0_39_dev20260401.so
I0000 00:00:1777019941.195875     339 pjrt_api.cc:96] PJRT_Api is set for device type tpu
I0000 00:00:1777019941.195878     339 pjrt_api.cc:167] The PJRT plugin has PJRT API version 0.103. The framework PJRT API version is 0.103.
I0000 00:00:1777019942.641143     339 pjrt_c_api_client.cc:182] PjRtCApiClient created.


In [8]:
res_Runtime = Dict(
    "KERNEL" => Sys.KERNEL,
    "MACHINE" => Sys.MACHINE,
    "CPU_NAME" => Sys.CPU_NAME,
    "JULIA_VERSION" => VERSION,
    "nthreads" => Threads.nthreads(),
    "TOTAL_MEMORY" => Sys.total_memory() |> Base.format_bytes,
    "FREE_MEMORY" => Sys.free_memory() |> Base.format_bytes
)

Dict{String, Any} with 7 entries:
  "nthreads"      => 24
  "KERNEL"        => :Linux
  "CPU_NAME"      => "znver3"
  "JULIA_VERSION" => v"1.12.6"
  "MACHINE"       => "x86_64-linux-gnu"
  "FREE_MEMORY"   => "40.281 GiB"
  "TOTAL_MEMORY"  => "47.049 GiB"

In [9]:
function comp_pi(x, y)
    inside = @. ifelse(x^2 + y^2 <= 1f0, 1f0, 0f0)
    return 4f0 * sum(inside) / Float32(length(x))
end

comp_pi (generic function with 1 method)

In [10]:
N = 10^8
x_host = rand(Float32, N)
y_host = rand(Float32, N)

comp_pi(x_host, y_host)

3.1418087f0

In [11]:
Sys.total_memory()

0x0000000bc322e000

In [13]:
b1 = @benchmark comp_pi(x_host, y_host)

BenchmarkTools.Trial: 34 samples with 1 evaluation per sample.
 Range (min … max):  143.272 ms … 159.907 ms  ┊ GC (min … max): 8.37% … 9.09%
 Time  (median):     147.899 ms               ┊ GC (median):    9.02%
 Time  (mean ± σ):   147.761 ms ±   3.087 ms  ┊ GC (mean ± σ):  9.02% ± 0.27%

    ▁     ▄█  ▁    ▁ ▄▄▁   ▁                                     
  ▆▁█▆▆▁▁▆██▁▁█▁▆▁▆█▆███▁▁▁█▁▆▁▆▁▆▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▆ ▁
  143 ms           Histogram: frequency by time          160 ms <

 Memory estimate: 381.47 MiB, allocs estimate: 4.

In [14]:
x_ra = Reactant.ConcreteRArray(x_host)
y_ra = Reactant.ConcreteRArray(y_host)
f = Reactant.compile(comp_pi, (x_ra, y_ra))
f(x_ra, y_ra)


ConcretePJRTNumber{Float32, 1}(3.141809f0)

In [15]:
b1_Reactant = @benchmark $f($x_ra, $y_ra)

BenchmarkTools.Trial: 4716 samples with 1 evaluation per sample.
 Range (min … max):  849.020 μs …  1.319 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):       1.058 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):     1.058 ms ± 20.367 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

                                               ▂█               
  ▂▁▂▁▁▂▁▁▁▁▁▁▂▁▁▂▂▂▁▁▂▂▂▂▂▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▃▄▄▆███▅▄▃▃▃▂▂▂▂▂▂▂ ▃
  849 μs          Histogram: frequency by time         1.12 ms <

 Memory estimate: 400 bytes, allocs estimate: 14.

In [16]:
b1_Reactant

BenchmarkTools.Trial: 4716 samples with 1 evaluation per sample.
 Range (min … max):  849.020 μs …  1.319 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):       1.058 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):     1.058 ms ± 20.367 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

                                               ▂█               
  ▂▁▂▁▁▂▁▁▁▁▁▁▂▁▁▂▂▂▁▁▂▂▂▂▂▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▃▄▄▆███▅▄▃▃▃▂▂▂▂▂▂▂ ▃
  849 μs          Histogram: frequency by time         1.12 ms <

 Memory estimate: 400 bytes, allocs estimate: 14.

In [17]:
b1

BenchmarkTools.Trial: 34 samples with 1 evaluation per sample.
 Range (min … max):  143.272 ms … 159.907 ms  ┊ GC (min … max): 8.37% … 9.09%
 Time  (median):     147.899 ms               ┊ GC (median):    9.02%
 Time  (mean ± σ):   147.761 ms ±   3.087 ms  ┊ GC (mean ± σ):  9.02% ± 0.27%

    ▁     ▄█  ▁    ▁ ▄▄▁   ▁                                     
  ▆▁█▆▆▁▁▆██▁▁█▁▆▁▆█▆███▁▁▁█▁▆▁▆▁▆▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▆ ▁
  143 ms           Histogram: frequency by time          160 ms <

 Memory estimate: 381.47 MiB, allocs estimate: 4.

Save benchmark details

In [18]:
res_Runtime

Dict{String, Any} with 7 entries:
  "nthreads"      => 24
  "KERNEL"        => :Linux
  "CPU_NAME"      => "znver3"
  "JULIA_VERSION" => v"1.12.6"
  "MACHINE"       => "x86_64-linux-gnu"
  "FREE_MEMORY"   => "40.281 GiB"
  "TOTAL_MEMORY"  => "47.049 GiB"

In [ ]:
b1

BenchmarkTools.Trial: 21 samples with 1 evaluation per sample.
 Range (min … max):  186.595 ms … 358.310 ms  ┊ GC (min … max): 6.30% … 0.00%
 Time  (median):     246.314 ms               ┊ GC (median):    5.61%
 Time  (mean ± σ):   245.666 ms ±  49.771 ms  ┊ GC (mean ± σ):  5.52% ± 2.00%

  ▁█▁▁█   █         ▁  ▁     ▁ ██▁ ▁           ▁ ▁            ▁  
  █████▁▁▁█▁▁▁▁▁▁▁▁▁█▁▁█▁▁▁▁▁█▁███▁█▁▁▁▁▁▁▁▁▁▁▁█▁█▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  187 ms           Histogram: frequency by time          358 ms <

 Memory estimate: 381.47 MiB, allocs estimate: 4.

In [19]:
res_Runtime["CPU_NAME"]

"znver3"

In [20]:

date_prefix = Dates.format(Dates.today(), "yyyy-mm-dd");
runtime_file = joinpath(date_prefix * "_TPU_" * res_Runtime["CPU_NAME"] * "_" * string(res_Runtime["nthreads"]) * "_" * "res_Runtime.jld2");
JLD2.jldsave(runtime_file, true; res_Runtime, b1, b1_Reactant);
println("Saved runtime data to: ", runtime_file);

Saved runtime data to: 2026-04-24_TPU_znver3_24_res_Runtime.jld2
